# Traffic Sign Recognition: Baseline CNN vs VGG19 Robustness Benchmark
This notebook trains your models on clean traffic sign images (GTSRB) and evaluates them across **14 real-world corruptions at 5 severity levels (70 conditions)** on a free Google Colab GPU.

### Hardware Setup:
1. Go to **Runtime > Change runtime type**.
2. Select **T4 GPU** as the Hardware accelerator.

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

### Step 2: Mount Google Drive & Set Project Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change directory to your uploaded project path:
%cd '/content/drive/MyDrive/Colab Notebooks/ML_Project_RegSwaTan'

### Step 3: Install Required Dependencies

In [ ]:
!pip install -r requirements.txt

### Step 4: Verify or Download Dataset (GTSRB)

In [ ]:
import os
dataset_dir = 'data/German Dataset'

if not os.path.exists(os.path.join(dataset_dir, 'Train.csv')):
    print('GTSRB dataset not detected. Downloading via kagglehub...')
    !pip install -q kagglehub
    import kagglehub
    path = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
    os.makedirs(dataset_dir, exist_ok=True)
    !cp -r "{path}"/* "{dataset_dir}/"

print('Dataset status: Ready!')
!ls -lh 'data/German Dataset/Train.csv'

### Step 5: Verify Models

In [ ]:
import torch
from models.RawCNN import get_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')

# Check Baseline CNN
cnn = get_model(num_classes=43, model_type='basic_cnn').to(device)
print(f'Baseline CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}')

# Check VGG19-BN
vgg = get_model(num_classes=43, model_type='vgg19_bn').to(device)
print(f'VGG19-BN parameters:     {sum(p.numel() for p in vgg.parameters()):,}')

### Step 6: Train Baseline CNN on Clean Images (~3 to 5 minutes)
Saves best model weights to `checkpoints/basic_cnn_german_best.pth`

In [ ]:
!python training/train.py --model_type basic_cnn --dataset german --epochs 25 --batch_size 64 -y

### Step 7: Train VGG19-BN on Clean Images (~15 to 20 minutes)
Saves best model weights to `checkpoints/vgg19_bn_german_best.pth`

In [ ]:
!python training/train.py --model_type vgg19_bn --dataset german --epochs 25 --batch_size 64 -y

### Step 8: Run 70-Condition Robustness Benchmark on Baseline CNN
Tests the trained CNN across **all 14 corruptions $\times$ 5 severities**.

In [ ]:
!python experiments/run_experiments.py --model_type basic_cnn --dataset german

### Step 9: Run 70-Condition Robustness Benchmark on VGG19-BN
Tests the trained VGG19-BN across **all 14 corruptions $\times$ 5 severities**.

In [ ]:
!python experiments/run_experiments.py --model_type vgg19_bn --dataset german

### Step 10: Preview Generated Results and Degradation Curves

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Display VGG19-BN 14x5 Summary Matrix
print('VGG19-BN Corruption Matrix (Accuracies % Across Severities 1 to 5):')
matrix_vgg = pd.read_csv('results/csv/robustness_vgg19_bn_german_matrix.csv')
display(matrix_vgg)

# Display Degradation Curves Figure
plot_path = 'results/plots/degradation_curves_vgg19_bn_german.png'
if os.path.exists(plot_path):
    print('\nPublication Degradation Curves (Noise, Blur, Weather, Digital):')
    display(Image(plot_path))